## Retrieve Order Data from Postgres Database

In [1]:
import pandas as pd
from sqlalchemy import create_engine

import json


In [2]:
with open("config.json", "r") as f:
    c_data = json.load(f)

USER = c_data["USER"]
PASSWORD = c_data["PASSWORD"]
HOST = c_data["HOST"]
PORT = c_data["PORT"]
DATABASE = c_data["DATABASE"]

conn_string = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(conn_string)

In [3]:
df_orders = pd.read_sql(f"SELECT * FROM orders_data;", engine)
print(df_orders.shape)
df_orders.head(5)

(185013, 9)


,Customer ID,Customer Status,Date Order was placed,Delivery Date,Order ID,Product ID,Quantity Ordered,Total Retail Price for This Order,Cost Price Per Unit
0,579,Silver,01-Jan-17,07-Jan-17,123002578,220101400106,2,92.6,20.70
1,7574,SILVER,01-Jan-17,05-Jan-17,123004074,210201000009,1,21.7,9.95
2,28861,Gold,01-Jan-17,04-Jan-17,123000871,230100500068,1,1.7,0.80
3,43796,Gold,01-Jan-17,06-Jan-17,123002851,220100100633,1,47.9,24.05
4,54673,Gold,01-Jan-17,04-Jan-17,123003607,220200200043,1,36.9,18.30


## Retrieve Product Data from Kaggle API

In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

c:\Users\admin\Documents\GitHub\ETL-Project-for-E-commerce-Sales-Data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df_product = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "gabrielsantello/wholesale-and-retail-orders-dataset", path="product-supplier.csv")

print(df_product.shape)
df_product.head(5)

(5504, 8)


,Product ID,Product Line,Product Category,Product Group,Product Name,Supplier Country,Supplier Name,Supplier ID
0,210100100001,Children,Children Outdoors,"Outdoor things, Kids",Boy's and Girl's Ski Pants with Braces,NO,Scandinavian Clothing A/S,50
1,210100100002,Children,Children Outdoors,"Outdoor things, Kids",Children's Jacket,ES,Luna sastreria S.A.,4742
2,210100100003,Children,Children Outdoors,"Outdoor things, Kids",Children's Jacket Sidney,NO,Scandinavian Clothing A/S,50
3,210100100004,Children,Children Outdoors,"Outdoor things, Kids",Children's Rain Set,NO,Scandinavian Clothing A/S,50
4,210100100005,Children,Children Outdoors,"Outdoor things, Kids",Children's Rain Suit,NO,Scandinavian Clothing A/S,50


## Clean Data

In [6]:
df_merged = pd.merge(df_orders, df_product, how="left", left_on="Product ID", right_on="Product ID")

In [7]:
print("Dataset shape:", df_merged.shape)
print("\nColumn names:", df_merged.columns.tolist())
print("\nData types:")
print(df_merged.dtypes)
print("\nMissing values:")
print(df_merged.isnull().sum())
print("\nDuplicate rows:")
print(df_merged.duplicated().sum())
df_merged.head(5)

Dataset shape: (185013, 16)

Column names: ['Customer ID', 'Customer Status', 'Date Order was placed', 'Delivery Date', 'Order ID', 'Product ID', 'Quantity Ordered', 'Total Retail Price for This Order', 'Cost Price Per Unit', 'Product Line', 'Product Category', 'Product Group', 'Product Name', 'Supplier Country', 'Supplier Name', 'Supplier ID']

Data types:
Customer ID                            int64
Customer Status                          str
Date Order was placed                    str
Delivery Date                            str
Order ID                               int64
Product ID                             int64
Quantity Ordered                       int64
Total Retail Price for This Order    float64
Cost Price Per Unit                  float64
Product Line                             str
Product Category                         str
Product Group                            str
Product Name                             str
Supplier Country                         str
Supplier N

,Customer ID,Customer Status,Date Order was placed,Delivery Date,Order ID,Product ID,Quantity Ordered,Total Retail Price for This Order,Cost Price Per Unit,Product Line,Product Category,Product Group,Product Name,Supplier Country,Supplier Name,Supplier ID
0,579,Silver,01-Jan-17,07-Jan-17,123002578,220101400106,2,92.6,20.70,Clothes & Shoes,Clothes,Tracker Clothes,Casual V-Neck Men's Sweatshirt,US,3Top Sports,2963
1,7574,SILVER,01-Jan-17,05-Jan-17,123004074,210201000009,1,21.7,9.95,Children,Children Sports,Tracker Kid's Clothes,Children's Tee,US,3Top Sports,2963
2,28861,Gold,01-Jan-17,04-Jan-17,123000871,230100500068,1,1.7,0.80,Outdoors,Outdoors,Outdoor Gear,Plate Picnic Deep,GB,Prime Sports Ltd,316
3,43796,Gold,01-Jan-17,06-Jan-17,123002851,220100100633,1,47.9,24.05,Clothes & Shoes,Clothes,Eclipse Clothing,Woman's Woven Pants L,US,Eclipse Inc,1303
4,54673,Gold,01-Jan-17,04-Jan-17,123003607,220200200043,1,36.9,18.30,Clothes & Shoes,Shoes,Shoes,Soft Gel Court Men's Indoor Shoes,US,Pro Sportswear Inc,1747


In [8]:
# Check for negative quantities/prices
print("Quantity <= 0:", (df_merged['Quantity Ordered'] <= 0).sum())
print("Revenue <= 0:", (df_merged['Total Retail Price for This Order'] <= 0).sum())

Quantity <= 0: 0
Revenue <= 0: 0


In [9]:
# Convert dates to datetime
df_merged['Date Order was placed'] = pd.to_datetime(df_merged['Date Order was placed'], format='%d-%b-%y')
df_merged['Delivery Date'] = pd.to_datetime(df_merged['Delivery Date'], format='%d-%b-%y')
df_merged.head(5)

,Customer ID,Customer Status,Date Order was placed,Delivery Date,Order ID,Product ID,Quantity Ordered,Total Retail Price for This Order,Cost Price Per Unit,Product Line,Product Category,Product Group,Product Name,Supplier Country,Supplier Name,Supplier ID
0,579,Silver,2017-01-01,2017-01-07,123002578,220101400106,2,92.6,20.70,Clothes & Shoes,Clothes,Tracker Clothes,Casual V-Neck Men's Sweatshirt,US,3Top Sports,2963
1,7574,SILVER,2017-01-01,2017-01-05,123004074,210201000009,1,21.7,9.95,Children,Children Sports,Tracker Kid's Clothes,Children's Tee,US,3Top Sports,2963
2,28861,Gold,2017-01-01,2017-01-04,123000871,230100500068,1,1.7,0.80,Outdoors,Outdoors,Outdoor Gear,Plate Picnic Deep,GB,Prime Sports Ltd,316
3,43796,Gold,2017-01-01,2017-01-06,123002851,220100100633,1,47.9,24.05,Clothes & Shoes,Clothes,Eclipse Clothing,Woman's Woven Pants L,US,Eclipse Inc,1303
4,54673,Gold,2017-01-01,2017-01-04,123003607,220200200043,1,36.9,18.30,Clothes & Shoes,Shoes,Shoes,Soft Gel Court Men's Indoor Shoes,US,Pro Sportswear Inc,1747


In [10]:
# Standardize customer tier
df_merged['Customer Status'] = df_merged['Customer Status'].str.title()
df_merged['Customer Status'].value_counts()

Customer Status
Silver      92541
Gold        88278
Platinum     4194
Name: count, dtype: int64

In [11]:
# Calculate profit per order
df_merged['Profit'] = df_merged['Total Retail Price for This Order'] - (df_merged['Quantity Ordered'] * df_merged['Cost Price Per Unit'])

# Calculate profit margin
df_merged['Profit Margin'] = (df_merged['Profit'] / df_merged['Total Retail Price for This Order']) * 100

# Calculate delivery time in days
df_merged['Delivery Days'] = (df_merged['Delivery Date'] - df_merged['Date Order was placed']).dt.days

df_merged.head(5)

,Customer ID,Customer Status,Date Order was placed,Delivery Date,Order ID,Product ID,Quantity Ordered,Total Retail Price for This Order,Cost Price Per Unit,Product Line,Product Category,Product Group,Product Name,Supplier Country,Supplier Name,Supplier ID,Profit,Profit Margin,Delivery Days
0,579,Silver,2017-01-01,2017-01-07,123002578,220101400106,2,92.6,20.70,Clothes & Shoes,Clothes,Tracker Clothes,Casual V-Neck Men's Sweatshirt,US,3Top Sports,2963,51.20,55.291577,6
1,7574,Silver,2017-01-01,2017-01-05,123004074,210201000009,1,21.7,9.95,Children,Children Sports,Tracker Kid's Clothes,Children's Tee,US,3Top Sports,2963,11.75,54.147465,4
2,28861,Gold,2017-01-01,2017-01-04,123000871,230100500068,1,1.7,0.80,Outdoors,Outdoors,Outdoor Gear,Plate Picnic Deep,GB,Prime Sports Ltd,316,0.90,52.941176,3
3,43796,Gold,2017-01-01,2017-01-06,123002851,220100100633,1,47.9,24.05,Clothes & Shoes,Clothes,Eclipse Clothing,Woman's Woven Pants L,US,Eclipse Inc,1303,23.85,49.791232,5
4,54673,Gold,2017-01-01,2017-01-04,123003607,220200200043,1,36.9,18.30,Clothes & Shoes,Shoes,Shoes,Soft Gel Court Men's Indoor Shoes,US,Pro Sportswear Inc,1747,18.60,50.406504,3


In [12]:
df_merged.to_sql("cleaned_orders_product_data", engine, if_exists="replace", index=False)

13